In [ ]:
import os
import time
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm
from sklearn.model_selection import train_test_split


# ==========================================================
# Dataset: matrices on-the-fly from FULL X_global time order
#   matrix(i) = [x_i, x_{i-1}, ...]  -> shape [F, W]
#   label(i)  = y_i
# ==========================================================
class SwatSeqFromFullDataset(Dataset):
    def __init__(
        self,
        X_full: np.ndarray,          # [N,F] in time order
        y_full: np.ndarray,          # [N]
        indices: np.ndarray,         # indices used by this split
        d_hist: int = 20,
        repeat_pattern: bool = True
    ):
        assert X_full.ndim == 2
        assert y_full.ndim == 1
        assert len(X_full) == len(y_full)

        self.X = X_full.astype(np.float32, copy=False)
        self.y = y_full.astype(np.int64, copy=False)
        self.idx = indices.astype(np.int64, copy=False)
        self.W = int(d_hist)
        self.repeat_pattern = bool(repeat_pattern)

    def __len__(self):
        return len(self.idx)

    def _make_matrix(self, i: int) -> np.ndarray:
        """
        Build [F,W] for original time index i:
          col0 = X[i]
          col1 = X[i-1]
          ...
        Fill if not enough history by repeating the available pattern (or padding with x_i).
        """
        x_i = self.X[i]       # [F]
        F = x_i.shape[0]
        W = self.W

        # collect history including current
        seq = [x_i]
        j = i - 1
        while j >= 0 and len(seq) < W:
            seq.append(self.X[j])
            j -= 1

        seq = np.stack(seq, axis=1)  # [F, L] L<=W

        if seq.shape[1] < W:
            if self.repeat_pattern:
                num_repeats = W // seq.shape[1]
                rem = W % seq.shape[1]
                out = np.tile(seq, (1, num_repeats))
                if rem > 0:
                    out = np.hstack([out, seq[:, :rem]])
            else:
                pad_cols = W - seq.shape[1]
                pad = np.tile(x_i.reshape(F, 1), (1, pad_cols))
                out = np.hstack([seq, pad])
        else:
            out = seq[:, :W]

        return out.astype(np.float32, copy=False)

    def __getitem__(self, k: int):
        i = int(self.idx[k])  # original time index in full series
        mat = self._make_matrix(i)        # [F,W]
        lab = int(self.y[i])              # scalar
        return torch.from_numpy(mat), torch.tensor(lab, dtype=torch.long)


# ==========================================================
# Mask-Guided Variable Window Pooling (same as your code)
# ==========================================================
class MaskGuidedVariablePooling(nn.Module):
    def __init__(self, K, tau=1.0):
        super().__init__()
        self.K = K
        self.tau = tau

        self.h_mlp = nn.Sequential(
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

        self.last_hk = None  # [B]
        self.last_m  = None  # [B,W]

    def forward(self, x):
        # x: [B, C, H, W]
        B, C, H, W = x.shape
        device = x.device

        pooled = x.mean(dim=[2,3])          # [B, C]
        h_k = self.h_mlp(pooled).squeeze(1) # [B]

        t = torch.linspace(0.0, 1.0, W, device=device).unsqueeze(0)  # [1,W]
        m = torch.sigmoid(self.tau * (h_k.unsqueeze(1) - t))         # [B,W]

        self.last_hk = h_k
        self.last_m  = m

        s = torch.relu(m[:, :-1] - m[:, 1:])  # [B,W-1]
        eps = 1e-8
        p = (s + eps) / (s.sum(dim=1, keepdim=True) + eps)  # [B,W-1]
        cdf = torch.cumsum(p, dim=1)                         # [B,W-1]
        cdf[:, -1] = 1.0

        quantiles = torch.linspace(0, 1, self.K+1, device=device)[1:-1]  # [K-1]
        cuts = []
        for q in quantiles:
            cut = torch.argmax((cdf >= q).int(), dim=1)  # [B]
            cuts.append(cut)
        cuts = torch.stack(cuts, dim=1) if len(cuts) else torch.empty(B,0,dtype=torch.long,device=device)

        outputs = []
        prev = torch.zeros(B, dtype=torch.long, device=device)
        idx = torch.arange(W, device=device).unsqueeze(0)  # [1,W]

        for r in range(self.K):
            if r < self.K-1:
                curr = cuts[:, r]
            else:
                curr = torch.full((B,), W-1, device=device, dtype=torch.long)

            seg_mask = ((idx >= prev.unsqueeze(1)) & (idx <= curr.unsqueeze(1))).float()
            weights = seg_mask * m
            weights = weights / (weights.sum(dim=1, keepdim=True) + eps)

            pooled_seg = (x * weights.unsqueeze(1).unsqueeze(2)).sum(dim=3)  # sum over W
            outputs.append(pooled_seg)
            prev = curr + 1

        return torch.stack(outputs, dim=3)  # [B,C,H,K]


# ==========================================================
# Regularization on hk
# ==========================================================
def hk_class_regularization(hk, y, eps=1e-8):
    unique = torch.unique(y)
    means = []
    intra = hk.new_tensor(0.0)
    valid_classes = 0

    for c in unique:
        mask = (y == c)
        if mask.sum() >= 2:
            hk_c = hk[mask]
            means.append(hk_c.mean())
            intra = intra + hk_c.var(unbiased=False)
            valid_classes += 1
        elif mask.sum() == 1:
            means.append(hk[mask].mean())
            valid_classes += 1

    L_intra = intra / (valid_classes + eps) if valid_classes > 0 else hk.new_tensor(0.0)

    if len(means) >= 2:
        means = torch.stack(means)
        diff = means.unsqueeze(0) - means.unsqueeze(1)
        L_inter = diff.pow(2).mean()
    else:
        L_inter = hk.new_tensor(0.0)

    return L_intra, L_inter


# ==========================================================
# Model (dynamic sizes)
# ==========================================================
class Model(nn.Module):
    def __init__(self, n_features: int, d_hist: int, n_classes: int = 2, tau: float = 1.0):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, (1, 3))
        self.pool1 = MaskGuidedVariablePooling(K=7, tau=tau)

        self.conv2 = nn.Conv2d(16, 16, (1, 2))
        self.pool2 = MaskGuidedVariablePooling(K=5, tau=tau)

        self.flatten = nn.Flatten()

        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_features, d_hist)
            dummy = torch.relu(self.conv1(dummy))
            dummy = self.pool1(dummy)
            dummy = torch.relu(self.conv2(dummy))
            dummy = self.pool2(dummy)
            flatten_size = dummy.numel()

        self.fc = nn.Sequential(
            nn.Linear(flatten_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.flatten(x)
        return self.fc(x)


# ==========================================================
# Train/Eval loops
# ==========================================================
@torch.no_grad()
def eval_accuracy(model, loader, device):
    model.eval()
    correct, total = 0, 0
    for xb, yb in loader:
        xb = xb.unsqueeze(1).to(device, non_blocking=True)  # [B,1,F,W]
        yb = yb.to(device, non_blocking=True)
        out = model(xb)
        pred = out.argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)
    return 100.0 * correct / max(1, total)


def train_one_setting(
    X_global: np.ndarray,
    y_global: np.ndarray,
    train_indices: np.ndarray,
    test_indices: np.ndarray,
    setting_name: str,
    d_hist: int = 20,
    repeat_pattern: bool = True,
    epochs: int = 30,
    batch_size: int = 256,
    lr: float = 5e-5,
    tau: float = 1.0,
    lambda_intra: float = 0.10,
    lambda_inter: float = 0.20,
    save_dir: str = "./swat_model"
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    N, F = X_global.shape
    print(f"\n================ {setting_name} ================")
    print("N,F:", (N, F), "| d_hist:", d_hist)
    print("Device:", device)
    print("Train size:", len(train_indices), "| Test size:", len(test_indices))

    train_ds = SwatSeqFromFullDataset(X_global, y_global, train_indices, d_hist=d_hist, repeat_pattern=repeat_pattern)
    test_ds  = SwatSeqFromFullDataset(X_global, y_global, test_indices,  d_hist=d_hist, repeat_pattern=repeat_pattern)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, pin_memory=True)

    model = Model(n_features=F, d_hist=d_hist, n_classes=2, tau=tau).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"model_swat_{setting_name}.pth")

    for epoch in range(epochs):
        model.train()
        t0 = time.time()

        running_loss = 0.0
        running_ce = 0.0
        running_reg = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f"{setting_name} | Epoch {epoch+1}/{epochs}", leave=False)
        for xb, yb in pbar:
            xb = xb.unsqueeze(1).to(device, non_blocking=True)  # [B,1,F,W]
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad()
            out = model(xb)

            ce = criterion(out, yb)

            hk1 = model.pool1.last_hk
            hk2 = model.pool2.last_hk
            L_intra1, L_inter1 = hk_class_regularization(hk1, yb)
            L_intra2, L_inter2 = hk_class_regularization(hk2, yb)

            reg = (L_intra1 + L_intra2) * lambda_intra - (L_inter1 + L_inter2) * lambda_inter
            loss = ce + reg

            loss.backward()
            optimizer.step()

            bs = yb.size(0)
            running_loss += loss.item() * bs
            running_ce += ce.item() * bs
            running_reg += reg.item() * bs

            pred = out.argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += bs

            pbar.set_postfix(loss=float(loss.item()), ce=float(ce.item()), reg=float(reg.item()))

        train_acc = 100.0 * correct / max(1, total)
        test_acc = eval_accuracy(model, test_loader, device)
        dt = time.time() - t0

        print(f"\n[{setting_name}] Epoch {epoch+1}/{epochs} | {dt:.1f}s")
        print(f"  Loss: {running_loss/total:.4f} (CE={running_ce/total:.4f} + REG={running_reg/total:.4f})")
        print(f"  Train acc: {train_acc:.2f}%")
        print(f"  Test  acc: {test_acc:.2f}%")

    torch.save(model.state_dict(), save_path)
    print("✔ Saved:", save_path)
    return model


# ==========================================================
# Split builders
# ==========================================================
def build_chronological_split_indices(N: int, split_ratio: float = 0.7):
    cut = int(N * split_ratio)
    train_idx = np.arange(0, cut, dtype=np.int64)
    test_idx  = np.arange(cut, N, dtype=np.int64)
    return train_idx, test_idx

def build_stratified_random_split_indices(y: np.ndarray, test_size: float = 0.3, seed: int = 843):
    all_idx = np.arange(len(y), dtype=np.int64)
    tr, te = train_test_split(all_idx, test_size=test_size, random_state=seed, stratify=y)
    return np.sort(tr), np.sort(te)  # sort not mandatory, but nice for debugging


# ==========================================================
# MAIN
# ==========================================================
if __name__ == "__main__":
    DATA_DIR = "./preprocessed_swat/"
    X_global = np.load(os.path.join(DATA_DIR, "X_global.npy"))  # [N,F]
    y_global = np.load(os.path.join(DATA_DIR, "y_global.npy"))  # [N]

    print("Loaded:", X_global.shape, y_global.shape)
    print("y counts:", {0: int((y_global==0).sum()), 1: int((y_global==1).sum())})

    # ---------- shared hyperparams ----------
    D_HIST = 20
    EPOCHS = 30
    BS = 256
    LR = 5e-5
    TAU = 1.0
    L_INTRA = 10
    L_INTER = 20
    REPEAT_PATTERN = True


    # ======================================================
    # Stratified random split (i.i.d. style)
    # ======================================================
    tr_idx_rand, te_idx_rand = build_stratified_random_split_indices(y_global, test_size=0.3, seed=843)
    train_one_setting(
        X_global=X_global,
        y_global=y_global,
        train_indices=tr_idx_rand,
        test_indices=te_idx_rand,
        setting_name="random_stratified",
        d_hist=D_HIST,
        repeat_pattern=REPEAT_PATTERN,
        epochs=EPOCHS,
        batch_size=BS,
        lr=LR,
        tau=TAU,
        lambda_intra=L_INTRA,
        lambda_inter=L_INTER,
        save_dir="./swat_model"
    )